In [6]:
from google.colab import files
uploaded = files.upload()

Saving products.csv to products.csv
Saving sales.csv to sales.csv


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,sum
spark = SparkSession.builder.appName("RetailProject").getOrCreate()

In [8]:
sales = spark.read.csv("/content/sales.csv",
                       header=True,
                       inferSchema=True)
products = spark.read.csv("/content/products.csv",
                          header=True,
                          inferSchema=True)

In [13]:
data = sales.join(products,"product_id")
data = data.withColumn(
    "total",
    col("quantity") * col("price")
)
data.show()

+----------+-------+--------+--------+----------+------------+-----------+-----+-----+------+
|product_id|sale_id|store_id|quantity| sale_date|product_name|   category|price| cost| total|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+------+
|         1|      6|     103|       1|2026-05-03|      Laptop|Electronics|50000|45000| 50000|
|         1|      1|     101|       2|2026-05-01|      Laptop|Electronics|50000|45000|100000|
|         2|      2|     101|       5|2026-05-01|       Mouse|Electronics|  500|  300|  2500|
|         3|      4|     102|       4|2026-05-02|    Keyboard|Electronics| 1000|  700|  4000|
|         4|      3|     102|       3|2026-05-02|       Shoes|    Fashion| 2000| 1500|  6000|
|         5|      5|     103|       6|2026-05-03|      Tshirt|    Fashion|  800|  500|  4800|
+----------+-------+--------+--------+----------+------------+-----------+-----+-----+------+



In [14]:
top = data.groupBy("product_name") \
          .agg(sum("total").alias("sales"))
top.show()

+------------+------+
|product_name| sales|
+------------+------+
|      Laptop|150000|
|       Mouse|  2500|
|      Tshirt|  4800|
|       Shoes|  6000|
|    Keyboard|  4000|
+------------+------+

